In [57]:
import requests

In [58]:
def read_github_repository(repo_owner, repo_name, branch="main"):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response = requests.get(url)
    response.raise_for_status()

    documents = []
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md', '.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc = {
                    'content': post.content,
                    'title': post.metadata.get('title'),
                    'description': post.metadata.get('description'),
                    'filename': file_path.split('/', 1)[-1]
                }
                documents.append(doc)

    return documents

In [81]:
def sliding_window(text, size=1000, step=500):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})

        start = end - step

        if end >= text_length:
            break

    return chunks

In [59]:
repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)

In [60]:
len(zip_response.content)

print(zip_response.status_code)
print(len(zip_response.content))

200
17544837


In [61]:
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [62]:
filenames = zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [63]:
filename = 'docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf-8')
print(mdx_content[:150])

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **


In [64]:
import frontmatter

post = frontmatter.loads(mdx_content)
print(post.content[:100])

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [65]:
post.metadata

{'title': 'Alerts', 'description': 'How to set up alerts.'}

In [66]:
filename_corrected = filename.split('/', 1)[-1]
print(filename_corrected)

docs/platform/alerts.mdx


In [67]:
doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

In [68]:
doc

{'content': '<Check>\n  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n![](/images/alerts.png)\n\nTo enable alerts, open the Project and navigate to the "Alerts" in the left menu. You must set:\n\n* A notification channel.\n\n* An alert condition.\n\n## Notification channels\n\nYou can choose between the following options:\n\n* **Email**. Add email addresses to send alerts to.\n\n* **Slack**. Add a Slack webhook.\n\n* **Discord**. Add a Discord webhook.\n\n## Alert conditions\n\n### Failed tests\n\nIf you use Tests (conditional checks) in your Project, you can tie alerting to the failed Tests in a Test Suite. Toggle this option on the Alerts page. Evidently will set an alert to the defined channel if any of the Tests fail.\n\n<Tip>\n  **How to avoid alert fatigue?** Use the `is_critical` parameter to mark non-critical Test as Warnings. Setting it to `False` prevent alerts for those checks even if they fail.\n</Tip>\n\n

In [69]:
repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")

Downloaded 95 documents


In [70]:
for doc in documents:
    print(doc["filename"])

api-reference/endpoint/create.mdx
api-reference/endpoint/delete.mdx
api-reference/endpoint/get.mdx
api-reference/introduction.mdx
changelog/changelog.mdx
docs/library/data_definition.mdx
docs/library/descriptors.mdx
docs/library/evaluations_overview.mdx
docs/library/leftover_content.mdx
docs/library/metric_generator.mdx
docs/library/output_formats.mdx
docs/library/overview.mdx
docs/library/prompt_optimization.mdx
docs/library/report.mdx
docs/library/synthetic_data_api.mdx
docs/library/tags_metadata.mdx
docs/library/tests.mdx
docs/platform/alerts.mdx
docs/platform/dashboard_add_panels.mdx
docs/platform/dashboard_add_panels_ui.mdx
docs/platform/dashboard_overview.mdx
docs/platform/dashboard_panel_types.mdx
docs/platform/datasets_generate.mdx
docs/platform/datasets_overview.mdx
docs/platform/datasets_workflow.mdx
docs/platform/evals_api.mdx
docs/platform/evals_explore.mdx
docs/platform/evals_no_code.mdx
docs/platform/evals_overview.mdx
docs/platform/monitoring_local_batch.mdx
docs/platfor

In [71]:
print(documents[9]["filename"])
print(documents[9]["content"])

docs/library/metric_generator.mdx
Sometimes you need to generate multiple column-level Tests or Metrics. To simplify this, you can use metric generator helper functions.

**Pre-requisites**:

* You know how to [generate Reports](/docs/library/report).

## Imports

<Accordion title="Generate data" defaultOpen={false}>
Use the following code to generate toy data for this guide.

```python
import pandas as pd
import numpy as np
from evidently import Dataset
from evidently import DataDefinition

np.random.seed(42)

data = {
    "Age": np.random.randint(18, 60, size=30),
    "Salary": np.random.randint(30000, 120000, size=30),
    "Department": np.random.choice(["HR", "IT", "Finance", "Marketing", "Operations"], size=30),
    "YearsExperience": np.random.randint(1, 15, size=30),  
    "EducationLevel": np.random.choice(["High School", "Bachelor", "Master", "PhD"], size=30)  
}

dummy_df = pd.DataFrame(data)

eval_data_1 = Dataset.from_pandas(
    dummy_df.iloc[:15],
    data_definition=Data

In [72]:
for doc in documents:
    print({
        'content': doc['content'][:150],
        'title': doc['title'],
        'description': doc['description'],
        'filename': doc['filename']
    })

{'content': '', 'title': 'Create Plant', 'description': None, 'filename': 'api-reference/endpoint/create.mdx'}
{'content': '', 'title': 'Delete Plant', 'description': None, 'filename': 'api-reference/endpoint/delete.mdx'}
{'content': '', 'title': 'Get Plants', 'description': None, 'filename': 'api-reference/endpoint/get.mdx'}
{'content': "<Note>\n  If you're not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## W", 'title': 'Introduction', 'description': 'Example section for showcasing API endpoints', 'filename': 'api-reference/introduction.mdx'}
{'content': '<Update label="2025-07-18" description="Evidently v0.7.11">\n  ## **Evidently 0.7.11**\n\n  Full release notes on [Github](https://github.com/evidentlyai', 'title': 'Product updates', 'description': 'Latest releases.', 'filename': 'changelog/changelog.mdx'}
{'content': 'To run evaluations, you must create a `Dataset` object with a `DataDefinition`, whi

In [73]:
from minsearch import Index

In [74]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [75]:
query = 'LLM as a Judge'
results = index.search(query, num_results=5)

In [76]:
len(results)

5

In [77]:
len(results[0]['content'])

21834

In [79]:
doc_sizes = [(doc['filename'], len(doc['content'])) for doc in documents]
doc_sizes.sort(key=lambda x: x[1], reverse=True)

for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")

metrics/all_metrics.mdx: 54996 characters
metrics/all_descriptors.mdx: 31874 characters
docs/platform/dashboard_panel_types.mdx: 31538 characters
docs/library/leftover_content.mdx: 28655 characters
metrics/customize_llm_judge.mdx: 26737 characters


In [80]:
document = list(range(0, 100))
window_size = 10
start = 0
step = 5

chunks = []

while start < len(document):
    end = start + window_size
    chunk = document[start:end]
    if len(chunk) < window_size:
        break
    chunks.append(chunk)
    print(chunk)
    start = start + step

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
[20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
[25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
[30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
[35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
[40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
[45, 46, 47, 48, 49, 50, 51, 52, 53, 54]
[50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
[55, 56, 57, 58, 59, 60, 61, 62, 63, 64]
[60, 61, 62, 63, 64, 65, 66, 67, 68, 69]
[65, 66, 67, 68, 69, 70, 71, 72, 73, 74]
[70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
[75, 76, 77, 78, 79, 80, 81, 82, 83, 84]
[80, 81, 82, 83, 84, 85, 86, 87, 88, 89]
[85, 86, 87, 88, 89, 90, 91, 92, 93, 94]
[90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [82]:
len(sliding_window(results[0]['content'], size=3000, step=2500))

39

In [83]:
documents[30]

{'content': 'AI observability lets you evaluate the quality of the inputs and outputs of your AI application as it runs in production. This gives an up-to-date view of your system behavior and helps spot and fix issues.\n\nEvidently offers several ways to set up monitoring.\n\n## Batch monitoring jobs\n\n<Check>\n  Supported in: `Evidently OSS`, `Evidently Cloud` and `Evidently Enterprise`.\n</Check>\n\n**Best for**: batch ML pipelines, regression testing, and near real-time ML systems that don’t need instant quality evaluations.\n\n![](/images/monitoring_flow_batch.png)\n\n**How it works**:\n\n* **Build your evaluation pipeline**. Create a pipeline in your infrastructure to run monitoring jobs. This can be a Python script, cron job, or orchestrated with a tool like Airflow. Run it at regular intervals (e.g., hourly, daily) or trigger it when new data or labels arrive.\n\n* **Run metric calculations**. Implement the evaluation step in the pipeline using the Evidently Python library. Se

In [84]:
document_chunks = []

for doc in documents:
    if not doc.get('content'):
        continue
    copy = doc.copy()
    content = copy.pop('content')

    chunks = sliding_window(content, size=3000, step=1500)

    for i, chunk in enumerate(chunks):
        chunk.update(copy)
        chunk['chunk_id'] = i
        document_chunks.append(chunk)

In [85]:
document_chunks[10]

{'start': 9000,
 'content': 'cation=[BinaryClassification(\n        target="target",\n        prediction_labels="prediction")],\n    categorical_columns=["target", "prediction"])\n```\n\nAvailable options and defaults:\n\n```python\n    target: str = "target"\n    prediction_labels: Optional[str] = None\n    prediction_probas: Optional[str] = "prediction" #if probabilistic classification\n    pos_label: Label = 1 #name of the positive label\n    labels: Optional[Dict[Label, str]] = None\n```\n\n### Ranking\n\n#### RecSys\n\nTo evaluate recommender systems performance, you must map the columns with:\n\n- Prediction: this could be predicted score or rank.\n- Target: relevance labels (e.g., this could be an interaction result like user click or upvote, or a true relevance label)\n\nThe **target** column can contain either:\n\n- a binary label (where `1` is a positive outcome)\n- any scores (positive values, where a higher value corresponds to a better match or a more valuable user action)

In [86]:
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [88]:
results = chunk_index.search(query)

[{'start': 0,
  'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create and tune the LLM evaluator**, which you can then apply in different cont

In [94]:
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, HTML

load_dotenv(override=True)

openai_client = OpenAI()

In [95]:
search_result = chunk_index.search(query, num_results=5)

In [96]:
query = 'how do I implement llm as a judge?'


In [97]:
import json

search_result_json = json.dumps(search_result, indent=2)

In [98]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

In [101]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [102]:
answer = llm(user_prompt, instructions)
print(answer)

To implement an LLM as a judge, follow these steps based on the tutorial provided:

1. **Installation**: Ensure you have the necessary Python library installed. Use the following command to install Evidently:
    ```bash
    pip install evidently
    ```

2. **Import Required Modules**: Import the necessary libraries in your Python script or Jupyter Notebook:
    ```python
    import pandas as pd
    import numpy as np
    from evidently import Dataset, Report, BinaryClassification
    from evidently.llm.templates import BinaryClassificationPromptTemplate
    ```

3. **Set Up OpenAI API Key**: Load your OpenAI API key as an environment variable:
    ```python
    import os
    os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
    ```

4. **Create Evaluation Dataset**: Prepare a dataset that contains:
   - **Questions**: Inputs to be evaluated.
   - **Target Responses**: Approved responses considered correct.
   - **New Responses**: The system-generated responses to be evaluated.
   - **Manual 

In [103]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [104]:
def search(query):
    return chunk_index.search(query, num_results=5)

In [105]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt

In [106]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [107]:
rag('how do I implement llm as a judge?')

'To implement an LLM (Large Language Model) as a judge, you can follow these steps based on the provided context:\n\n1. **Set Up Your Environment**:\n   - Install the necessary Python library, Evidently, using:\n     ```bash\n     pip install evidently\n     ```\n\n2. **Import Required Modules**:\n   - In your Python script or Jupyter notebook, import the relevant modules:\n     ```python\n     import os\n     import pandas as pd\n     from evidently import Dataset, Report, TextEvals\n     from evidently.llm.templates import BinaryClassificationPromptTemplate\n     ```\n\n3. **Set Your OpenAI API Key**:\n   - Set your OpenAI API key as an environment variable:\n     ```python\n     os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n     ```\n\n4. **Create Your Evaluation Dataset**:\n   - Prepare a dataset that includes:\n     - **Questions**: Inputs for the LLM.\n     - **Target responses**: Approved accurate responses.\n     - **New responses**: Responses you want to evaluate.\n     - **Manua